# VolatilityLab-BTC
**MScFE 610 Financial Econometrics, Group Work Project 1 (Group 8665)**

Four challenges are covered: (1) skewness, (2) kurtosis and heteroscedasticity, (3) sensitivity to outliers, (4) non-stationarity and unit-root testing. Each uses real Bitcoin returns and a simulated normal benchmark.

**How to run:** choose *Runtime > Run all* (Colab) or *Kernel > Restart & Run All* (Jupyter). If Yahoo Finance is unreachable, the notebook falls back to a cached `btc_close.csv` in the same folder.

In [ ]:
%pip install -q yfinance arch

## Setup and data

In [ ]:
import warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import skew, skewtest, kurtosis, jarque_bera
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import het_arch
from statsmodels.graphics.tsaplots import plot_acf
from arch import arch_model

warnings.filterwarnings("ignore")
sns.set_theme()

TICKER, START, END, SEED = "BTC-USD", "2020-01-01", "2025-03-15", 42   # fixed end date => reproducible results
ANN = 252   # annualisation factor used throughout (use 365 if you prefer 24/7 crypto convention)

def load_close():
    """Download BTC closes; fall back to a cached CSV if the download fails."""
    try:
        import yfinance as yf
        df = yf.download(TICKER, start=START, end=END, progress=False, auto_adjust=True)
        close = df["Close"]
        if isinstance(close, pd.DataFrame):      # newer yfinance returns a one-column frame
            close = close.iloc[:, 0]
        close = close.dropna()
        if len(close) < 500:
            raise ValueError("too few rows downloaded")
        close.name = "Close"
        close.to_csv("btc_close.csv")
        return close
    except Exception as err:
        print("Download failed (%s). Using cached btc_close.csv" % err)
        return pd.read_csv("btc_close.csv", index_col=0, parse_dates=True).iloc[:, 0]

close = load_close()
returns = close.pct_change().dropna()
returns.name = "Daily_Return"

np.random.seed(SEED)
sim_returns = pd.Series(np.random.normal(returns.mean(), returns.std(), len(returns)), index=returns.index)  # same length as BTC
np.random.seed(SEED)
sim_10k = np.random.normal(returns.mean(), returns.std(), 10_000)                                          # large sample for skewness

print(f"{len(returns)} daily returns from {returns.index[0].date()} to {returns.index[-1].date()}")

## Challenge 1: Skewness

In [ ]:
# Q1 - skewness of BTC returns vs simulated normal returns
skew_btc, skew_sim = skew(returns), skew(sim_10k)          # g1 = m3 / m2**1.5
z_btc, p_btc = skewtest(returns)                           # D'Agostino test, H0: symmetric
z_sim, p_sim = skewtest(sim_10k)
print(f"BTC skewness       : {skew_btc:.4f}  (z = {z_btc:.2f}, p = {p_btc:.4g})")
print(f"Simulated skewness : {skew_sim:.4f}  (z = {z_sim:.2f}, p = {p_sim:.4g})")

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(returns, bins=40, kde=True, ax=ax[0])
ax[0].axvline(returns.mean(), color="g", ls="--", label="Mean")
ax[0].axvline(returns.median(), color="b", label="Median")
ax[0].set(title="BTC daily returns", xlabel="Daily return", ylabel="Frequency"); ax[0].legend()
sns.histplot(sim_10k, bins=40, kde=True, ax=ax[1])
ax[1].set(title="Simulated normal returns (10,000 draws)", xlabel="Daily return", ylabel="Frequency")
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
(returns.rolling(20).std() * np.sqrt(ANN)).plot(ax=ax[0], title="BTC 20-day annualised volatility")
returns.rolling(250).skew().plot(ax=ax[1], title="BTC rolling 250-day skewness")
ax[0].set(xlabel="Date", ylabel="Volatility"); ax[1].set(xlabel="Date", ylabel="Skewness")
plt.tight_layout(); plt.show()

**Interpretation.** A clearly negative skewness with a tiny p-value means large daily losses are more likely than equally large gains, whereas the simulated normal series is symmetric. Symmetric models will understate downside risk, so use skew-aware models (see Directions in the report).

## Challenge 2: Kurtosis and heteroscedasticity

In [ ]:
# Q2 - excess kurtosis, Jarque-Bera, ARCH-LM and GARCH(1,1)
def tails_and_arch(x, label):
    ek = kurtosis(x)                      # SciPy default = EXCESS kurtosis (normal = 0)
    jb_stat, jb_p = jarque_bera(x)
    lm_stat, lm_p, _, _ = het_arch(x)
    res = arch_model(x * 100, vol="Garch", p=1, q=1).fit(disp="off")     # x100 = percent scale for the optimiser
    res_t = arch_model(x * 100, vol="Garch", p=1, q=1, dist="t").fit(disp="off")
    print(f"{label}: excess kurtosis = {ek:.3f} (raw = {ek + 3:.3f}) | Jarque-Bera p = {jb_p:.4g} | ARCH-LM p = {lm_p:.4g}")
    print(f"   GARCH(1,1) normal  : alpha = {res.params['alpha[1]']:.3f}, beta = {res.params['beta[1]']:.3f}, AIC = {res.aic:.1f}")
    print(f"   GARCH(1,1) Student-t: nu = {res_t.params['nu']:.2f}, AIC = {res_t.aic:.1f}")
    fig, ax = plt.subplots(4, 1, figsize=(12, 14))
    sns.kdeplot(x, ax=ax[0]); ax[0].set(title=f"{label} returns distribution (excess kurtosis {ek:.3f})", xlabel="Return")
    ax[1].scatter(x.shift(1), x, alpha=0.5)
    ax[1].set(title=f"{label}: return vs previous return (ARCH-LM p = {lm_p:.3f})", xlabel="Previous return", ylabel="Current return")
    plot_acf(x ** 2, lags=30, ax=ax[2], title=f"{label}: autocorrelation of squared returns"); ax[2].set_xlabel("Lag")
    ax[3].plot(x.index, res.conditional_volatility / 100); ax[3].set(title=f"{label} GARCH(1,1) conditional volatility", xlabel="Date", ylabel="Daily volatility")
    plt.tight_layout(); plt.show()
    return {"excess_kurtosis": ek, "arch_lm_p": lm_p}

k_btc = tails_and_arch(returns, "BTC")
k_sim = tails_and_arch(sim_returns, "Simulated")

**Interpretation.** Large positive excess kurtosis and an ARCH-LM p-value below 0.05 show fat tails and volatility clustering in Bitcoin. The simulated series shows neither. A Student-t GARCH(1,1) is the sensible baseline (lower AIC than the normal version indicates the fat tails matter).

## Challenge 3: Sensitivity to outliers

In [ ]:
# Q3 - rolling MAD outlier flags and MAD vs standard-deviation volatility
W = 20
d = pd.DataFrame({"r": returns})
d["median"] = d["r"].rolling(W).median()
d["mad"] = (d["r"] - d["median"]).abs().rolling(W).median()
d["scaled_mad"] = 1.4826 * d["mad"]                       # consistent with sigma under normality
d["upper"] = d["median"] + 3 * d["scaled_mad"]
d["lower"] = d["median"] - 3 * d["scaled_mad"]
d = d.dropna()
d["outlier"] = (d["r"] > d["upper"]) | (d["r"] < d["lower"])
d["vol_std"] = d["r"].rolling(W).std() * np.sqrt(ANN)
d["vol_mad"] = d["scaled_mad"] * np.sqrt(ANN)
print(f"Flagged outliers: {d['outlier'].sum()} of {len(d)} days ({d['outlier'].mean():.1%}); normal data would flag about 0.3% of days")

plt.figure(figsize=(14, 6))
plt.scatter(d.index, d["r"], c=d["outlier"].map({True: "red", False: "blue"}), alpha=0.5)
plt.title("BTC daily returns with outliers highlighted in red"); plt.xlabel("Date"); plt.ylabel("Daily return"); plt.show()

plt.figure(figsize=(14, 6))
plt.plot(d["vol_mad"], color="red", label="MAD-based volatility")
plt.plot(d["vol_std"], color="green", ls="--", label="Standard-deviation volatility")
plt.title("BTC volatility: MAD vs standard deviation"); plt.xlabel("Date"); plt.ylabel("Annualised volatility"); plt.legend(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(d["r"], bins=50, kde=True, ax=ax[0]); ax[0].set(title="All returns", xlabel="Daily return")
sns.histplot(d.loc[~d["outlier"], "r"], bins=50, kde=True, ax=ax[1]); ax[1].set(title="Flagged outliers removed (for illustration only)", xlabel="Daily return")
plt.tight_layout(); plt.show()
print("Std dev of all returns :", round(d["r"].std(), 5), "| after removing flags:", round(d.loc[~d["outlier"], "r"].std(), 5))

**Interpretation.** The MAD-based volatility reacts less to single extreme days than the standard deviation. Removing flagged days narrows the distribution, but those days are real crashes and rallies: use the robust measure for *typical* volatility and keep the extremes in tail-risk measures.

## Challenge 4: Non-stationarity and unit roots

In [ ]:
# Q4 - ADF (H0: unit root) and KPSS (H0: stationary), constant-only specification
def stationarity(price, ret, label):
    out = {"price_adf_p": adfuller(price)[1], "price_kpss_p": kpss(price, regression="c", nlags="auto")[1],
           "ret_adf_p": adfuller(ret)[1], "ret_kpss_p": kpss(ret, regression="c", nlags="auto")[1]}
    fig, ax = plt.subplots(2, 1, figsize=(12, 8))
    ax[0].plot(price); ax[0].set_title(f"{label} price | ADF p = {out['price_adf_p']:.4f} | KPSS p = {out['price_kpss_p']:.4f}")
    ax[1].plot(ret);   ax[1].set_title(f"{label} returns | ADF p = {out['ret_adf_p']:.4f} | KPSS p = {out['ret_kpss_p']:.4f}")
    for a in ax: a.set_xlabel("Date")
    ax[0].set_ylabel("Price"); ax[1].set_ylabel("Return"); plt.tight_layout(); plt.show()
    return out

sim_price = pd.Series(np.exp(np.cumsum(sim_returns)), index=sim_returns.index)    # simulated returns treated as log returns
s_btc = stationarity(close.loc[returns.index], returns, "BTC")
s_sim = stationarity(sim_price, sim_returns, "Simulated")
print("KPSS p-values are read from a table: 0.01 means p <= 0.01 and 0.10 means p >= 0.10.")

**Interpretation.** For prices, ADF fails to reject a unit root while KPSS rejects stationarity, so prices are non-stationary. Returns pass both tests. Model returns, not prices.

## Summary table

In [ ]:
summary = pd.DataFrame({
    "Bitcoin": [skew_btc, k_btc["excess_kurtosis"], k_btc["arch_lm_p"], s_btc["price_adf_p"], s_btc["price_kpss_p"], s_btc["ret_adf_p"], s_btc["ret_kpss_p"]],
    "Simulated normal": [skew_sim, k_sim["excess_kurtosis"], k_sim["arch_lm_p"], s_sim["price_adf_p"], s_sim["price_kpss_p"], s_sim["ret_adf_p"], s_sim["ret_kpss_p"]]},
    index=["Skewness", "Excess kurtosis", "ARCH-LM p-value", "Price ADF p", "Price KPSS p", "Returns ADF p", "Returns KPSS p"])
summary.round(4).to_csv("results.csv")
summary.round(4)